# RMSX Molstar Clean Import Test

This is the notebook we want once the Molstar work is committed/pushed into the main RMSX repo or a GitHub branch. It installs/imports RMSX normally, unpacks real RMSX test slice PDBs, and renders Molstar inline. No embedded viewer fallback.


In [ ]:
from pathlib import Path
import json
import sys

IN_COLAB = "google.colab" in sys.modules
OUTPUT_ROOT = Path("/content/rmsx_molstar_clean_import") if IN_COLAB else Path("rmsx_molstar_clean_import")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Change this to the branch/tag/commit that contains the Molstar files.
# Examples:
# GITHUB_REF = "rmsx_molstar"
# GITHUB_REF = "codex/molstar-notebook"
# GITHUB_REF = "<commit-sha>"
GITHUB_REF = "rmsx_molstar"
INSTALL_FROM_GITHUB = IN_COLAB

if INSTALL_FROM_GITHUB:
    %pip install -q --upgrade "git+https://github.com/AntunesLab/rmsx.git@{GITHUB_REF}"

print("Running in Colab:", IN_COLAB)
print("Output root:", OUTPUT_ROOT.resolve())


In [ ]:
import rmsx
from rmsx import build_molstar_manifest, write_molstar_flipbook, run_flipbook

print("rmsx:", Path(rmsx.__file__).resolve())
print("Molstar helpers imported cleanly.")


## Unpack Real Test Fixtures

These are real RMSX/Flipbook slice PDB outputs from the repo: 1UBQ and protease combined.


In [ ]:
import base64
import io
import zipfile

REAL_FIXTURE_ZIP_B64 = """..."""
REAL_FIXTURE_ROOT = OUTPUT_ROOT / "real_test_fixtures"
REAL_FIXTURE_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(io.BytesIO(base64.b64decode(REAL_FIXTURE_ZIP_B64))) as archive:
    archive.extractall(REAL_FIXTURE_ROOT)

fixture_dirs = {
    "1UBQ 9-slice RMSX": REAL_FIXTURE_ROOT / "example_uqb_chain_7_rmsx",
    "Protease combined 9-slice RMSX": REAL_FIXTURE_ROOT / "protease_combined",
}

for label, path in fixture_dirs.items():
    print(label, len(list(path.glob("slice_*_first_frame.pdb"))), "slices", path)


## Manifest Check


In [ ]:
for label, path in fixture_dirs.items():
    manifest = build_molstar_manifest(path, palette="mako")
    print("\n" + label)
    print(json.dumps({
        "sliceCount": len(manifest["slices"]),
        "residueCount": len(manifest["residues"]),
        "domain": manifest["domain"],
        "cameraMode": manifest["molstarRenderStyle"].get("cameraMode"),
        "maskedResidues": manifest["maskSummary"].get("maskedResidues"),
    }, indent=2))


## Render Real Fixtures


In [ ]:
from IPython.display import HTML, display

render_cases = [
    ("1UBQ 9-slice RMSX", fixture_dirs["1UBQ 9-slice RMSX"], "viridis", 1.0, 620),
    ("Protease combined 9-slice RMSX", fixture_dirs["Protease combined 9-slice RMSX"], "turbo", 0.85, 760),
]

for label, path, palette, spacing, height in render_cases:
    result = write_molstar_flipbook(
        path,
        palette=palette,
        spacing_factor=spacing,
        camera_mode="orthographic",
        output_html=OUTPUT_ROOT / f"{label.lower().replace(' ', '_').replace('-', '_')}.html",
        output_manifest=OUTPUT_ROOT / f"{label.lower().replace(' ', '_').replace('-', '_')}.json",
        asset_mode="cdn",
        iframe_height=height,
    )
    display(HTML(f"<h4>{label}: {palette}, orthographic</h4>"))
    display(result)
    print(result.html_path)


## Protease Camera Comparison


In [ ]:
protease_dir = fixture_dirs["Protease combined 9-slice RMSX"]
for mode in ["orthographic", "perspective"]:
    result = write_molstar_flipbook(
        protease_dir,
        palette="mako",
        camera_mode=mode,
        output_html=OUTPUT_ROOT / f"protease_camera_{mode}.html",
        output_manifest=OUTPUT_ROOT / f"protease_camera_{mode}.json",
        asset_mode="cdn",
        iframe_height=720,
    )
    display(HTML(f"<h4>Protease camera_mode={mode}</h4>"))
    display(result)
    print(mode, result.manifest["molstarRenderStyle"]["cameraMode"])
